In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("clean_data.csv")

In [3]:
df['date'] = pd.to_datetime(df['date'],format='%Y-%m-%d')
df = df.set_index('date')

In [4]:
x = df.drop(['price','total_duration_mins'],axis=1)
y = df[['total_duration_mins','price']]

In [5]:
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error,root_mean_squared_error

In [6]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42,shuffle=True)

In [7]:
import joblib

ct = joblib.load('ct.joblib')

In [8]:
x_train = ct.transform(x_train)
x_test = ct.transform(x_test)

In [9]:
import optuna

In [10]:

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0), 
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),    
        'gamma': trial.suggest_float('gamma', 1e-3, 10.0, log=True),
        'random_state': 42,
        'objective':'count:poisson',
        'n_jobs': -1  #
    }

    model = XGBRegressor(**params)
    model = MultiOutputRegressor(model)
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    score = r2_score(y_test, y_pred)
    return score


study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=15)

print('Best hyperparameters:', study.best_params)
print('Best score:', study.best_value)

[I 2026-06-30 13:53:33,631] A new study created in memory with name: no-name-6d9b7018-a5d4-4445-82cf-a6ff178a10b3
[I 2026-06-30 13:54:07,968] Trial 0 finished with value: 0.7594318389892578 and parameters: {'n_estimators': 250, 'max_depth': 10, 'learning_rate': 0.05395030966670229, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'min_child_weight': 2, 'gamma': 0.0017073967431528124}. Best is trial 0 with value: 0.7594318389892578.
[I 2026-06-30 13:54:45,600] Trial 1 finished with value: 0.760491132736206 and parameters: {'n_estimators': 450, 'max_depth': 7, 'learning_rate': 0.051059032093947576, 'subsample': 0.5102922471479012, 'colsample_bytree': 0.9849549260809971, 'min_child_weight': 9, 'gamma': 0.0070689749506246055}. Best is trial 1 with value: 0.760491132736206.
[I 2026-06-30 13:54:49,504] Trial 2 finished with value: 0.6304395198822021 and parameters: {'n_estimators': 150, 'max_depth': 4, 'learning_rate': 0.02014847788415866, 'subsample': 0.7623782158161

Best hyperparameters: {'n_estimators': 500, 'max_depth': 8, 'learning_rate': 0.09941810361139941, 'subsample': 0.5043736834843053, 'colsample_bytree': 0.8329012695800526, 'min_child_weight': 10, 'gamma': 0.0018753480452096743}
Best score: 0.7679661512374878


In [11]:
import mlflow
import mlflow.sklearn
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("flight_price")
model = XGBRegressor(**study.best_params,objective='count:poisson')
model = MultiOutputRegressor(model)
model.fit(x_train, y_train)
y_pred = model.predict(x_test)
scores = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

with mlflow.start_run(run_name="XGBRegressor_MultiOutputRegressor_model"):
    mlflow.log_params(study.best_params)
    mlflow.log_metric("r2_score", scores)
    mlflow.log_metric("mean_absolute_error", mae)
    mlflow.log_metric("mean_squared_error", mse)
    mlflow.log_metric("root_mean_squared_error", rmse)
    
    mlflow.sklearn.log_model(model, artifact_path="XGBRegressor_MultiOutputRegressor_model") 

2026/06/30 14:13:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/30 14:13:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBRegressor_MultiOutputRegressor_model at: http://127.0.0.1:5000/#/experiments/1/runs/c18f160bb23d4d63bc34c65026389a10
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [12]:
joblib.dump(model,'XGBRegressor_MultiOutputRegressor_model.joblib')

['XGBRegressor_MultiOutputRegressor_model.joblib']

In [13]:

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500, step=50), 
        'max_depth': trial.suggest_int('max_depth', 3, 10),                
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'max_features': trial.suggest_float('max_features', 0.5, 0.9),       
        'max_samples': trial.suggest_float('max_samples', 0.5, 0.8),        
        'n_jobs': -1,                                                        
        'random_state': 42
    }

    model = RandomForestRegressor(**params)
    model = MultiOutputRegressor(model)
    
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    score = r2_score(y_test, y_pred)
    
    return score

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))

study.optimize(objective, n_trials=15)

print('Best hyperparameters:', study.best_params)
print('Best score:', study.best_value)


[I 2026-06-30 14:13:55,532] A new study created in memory with name: no-name-c4ba2448-40e0-4797-86b2-7a1c3ac79d86
[I 2026-06-30 14:15:20,180] Trial 0 finished with value: 0.7389120141628993 and parameters: {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 15, 'min_samples_leaf': 12, 'max_features': 0.5624074561769746, 'max_samples': 0.5467983561008608}. Best is trial 0 with value: 0.7389120141628993.
[I 2026-06-30 14:15:36,822] Trial 1 finished with value: 0.7315887401072634 and parameters: {'n_estimators': 50, 'max_depth': 9, 'min_samples_split': 13, 'min_samples_leaf': 15, 'max_features': 0.508233797718321, 'max_samples': 0.7909729556485983}. Best is trial 0 with value: 0.7389120141628993.
[I 2026-06-30 14:16:18,325] Trial 2 finished with value: 0.6837377930361886 and parameters: {'n_estimators': 450, 'max_depth': 4, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 0.621696897183815, 'max_samples': 0.6574269294896714}. Best is trial 0 with value: 0.73891201416

Best hyperparameters: {'n_estimators': 50, 'max_depth': 10, 'min_samples_split': 6, 'min_samples_leaf': 14, 'max_features': 0.6246844304357644, 'max_samples': 0.6560204063533432}
Best score: 0.7390887625473639


In [14]:
model = RandomForestRegressor(**study.best_params)
model = MultiOutputRegressor(model)
model.fit(x_train, y_train)
y_pred = model.predict(x_test)
scores = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

with mlflow.start_run(run_name="RandomForestRegressor_MultiOutputRegressor_model"):
    mlflow.log_params(study.best_params)
    mlflow.log_metric("r2_score", scores)
    mlflow.log_metric("mean_absolute_error", mae)
    mlflow.log_metric("mean_squared_error", mse)
    mlflow.log_metric("root_mean_squared_error", rmse)
    mlflow.sklearn.log_model(model, artifact_path="RandomForestRegressor_MultiOutputRegressor_model")


2026/06/30 14:26:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/30 14:26:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RandomForestRegressor_MultiOutputRegressor_model at: http://127.0.0.1:5000/#/experiments/1/runs/f0bc028a6bff49f8b7793e03b6b92203
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [15]:
joblib.dump(model,'RandomForestRegressor_MultiOutputRegressor_model.joblib')

['RandomForestRegressor_MultiOutputRegressor_model.joblib']